# Session 5 — Rich-Club Organisation

**Goal of this session:** ask a very specific structural question — do the highest-degree nodes preferentially wire up with *each other*, more than their degrees alone would predict?

*Network Neuroscience in Python, session 5 of 10.*

## Why this matters

Session 4 classified individual nodes by hub type. This session asks about the hubs *as a group*. If the highest-degree regions of a network form a tightly interconnected core — a "rich club" — that has real consequences: it means the most influential nodes aren't just individually well-connected, they're organised into a backbone that most signal traffic likely passes through. This has been reported repeatedly in human connectomes and is one of the more replicated findings in the field. It is also, as we'll see, a measurement that is close to meaningless without a null model — which is why it comes right after sessions 1 and 2, not before them.

## The toy network, again

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """Same function as session 1. Not a brain."""
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)
    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])
    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)
    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))
    return G


def degree_preserving_null(G, n_swaps_per_edge=10, seed=0):
    """Same degree-preserving randomisation as sessions 1-2."""
    rng = np.random.default_rng(seed)
    G_null = G.copy()
    n_swaps = max(10, n_swaps_per_edge * G.number_of_edges())
    nx.double_edge_swap(G_null, nswap=n_swaps, max_tries=n_swaps * 20,
                         seed=int(rng.integers(1_000_000_000)))
    return G_null


G = generate_toy_network(seed=0)
degrees = [d for _, d in G.degree()]
print(f"{G.number_of_nodes()} nodes, degree range {min(degrees)}-{max(degrees)}")

## The rich-club coefficient

For a degree threshold *k*, take only the nodes with degree strictly greater than *k*, count how many edges actually connect pairs within that subgroup, and divide by the maximum possible number of edges among them:

$$\phi(k) = \frac{2 E_{>k}}{N_{>k}(N_{>k}-1)}$$

`networkx` computes this for every threshold in one call. Higher values of $\phi(k)$ mean the high-degree "rich" nodes are more densely interconnected among themselves.

In [ ]:
rc_raw = nx.rich_club_coefficient(G, normalized=False)
for k, phi in rc_raw.items():
    n_above = sum(1 for d in degrees if d > k)
    print(f"k={k}   nodes with degree>k: {n_above:2d}   phi(k)={phi:.3f}")

## Why the raw number is not enough

Here's the trap: $\phi(k)$ **mechanically increases with $k$** in almost any network, for a boring reason that has nothing to do with organisation. As you raise the threshold, you keep only the highest-degree nodes — and those nodes, by definition, have more edges to spend, so they're more likely to reach each other purely by chance. A raw rich-club coefficient rising with degree threshold is not, on its own, evidence of a rich club. You already built the machinery to fix this, back in session 1: compare against a degree-preserving null.

In [ ]:
def rich_club_null_curve(G, ks, n_random=300, seed=0):
    """Average rich-club coefficient at each k, across many degree-preserving nulls."""
    rng = np.random.default_rng(seed)
    curves = np.full((n_random, len(ks)), np.nan)
    for i in range(n_random):
        G_null = degree_preserving_null(G, seed=int(rng.integers(1_000_000_000)))
        rc_null = nx.rich_club_coefficient(G_null, normalized=False)
        for j, k in enumerate(ks):
            if k in rc_null:
                curves[i, j] = rc_null[k]
    return np.nanmean(curves, axis=0)


ks = sorted(rc_raw.keys())
phi_obs = np.array([rc_raw[k] for k in ks])
phi_null = rich_club_null_curve(G, ks, n_random=300, seed=1)
phi_normalized = phi_obs / phi_null

for k, o, nl, r in zip(ks, phi_obs, phi_null, phi_normalized):
    print(f"k={k}   observed={o:.3f}   null mean={nl:.3f}   normalized phi={r:.3f}")

## The real versus the normalised curve

$\phi_{\text{norm}}(k) > 1$ means the real network's high-degree nodes are *more* interconnected than a degree-matched random network predicts — genuine rich-club organisation. $\phi_{\text{norm}}(k) \approx 1$ means degree alone explains what we saw.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ks, phi_obs, "o-", color="#2b6cb0", label="observed", linewidth=2)
axes[0].plot(ks, phi_null, "s--", color="#a0aec0", label="null mean (degree-preserving)", linewidth=2)
axes[0].set_xlabel("degree threshold k", fontsize=12)
axes[0].set_ylabel("rich-club coefficient φ(k)", fontsize=12)
axes[0].set_title("Raw φ(k) always climbs with k", fontsize=12)
axes[0].legend(fontsize=10)

axes[1].plot(ks, phi_normalized, "o-", color="#c53030", linewidth=2)
axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("degree threshold k", fontsize=12)
axes[1].set_ylabel("normalised φ(k) = φ_real / φ_null", fontsize=12)
axes[1].set_title("Normalised: >1 means real rich-club organisation", fontsize=12)

plt.tight_layout()
plt.show()

## Reading the result honestly

At the lowest degree thresholds, normalised φ sits close to 1 — most of the network's nodes aren't unusually interconnected once you account for their degree. At the highest thresholds it climbs above 1, suggesting the very highest-degree nodes (which, by construction, includes the connector we built) are more tightly wired together than chance predicts.

One caveat worth stating plainly, because it applies to any small network including some real single-subject connectomes: at the very highest thresholds, only a couple of nodes remain above the cutoff. With two nodes there is exactly one possible edge, so φ saturates at 0 or 1 trivially and stops being informative — that tail of the curve should be read with far less confidence than the middle of it.

**Next session:** the single most common real mistake in this literature — comparing a graph metric between two groups whose networks differ only in density, not in organisation.